# EE-411 — Fundamentals of Inference and Learning
## Exercise Session 1 — Resampling and Statistical Inference

**Learning objectives**
- Manipulate and summarize data with pandas.
- Compare statistics between populations.
- Implement and interpret a one-sided permutation test.
- Use bootstrap resampling to estimate uncertainty in sample statistics.
- Apply these methods independently to the Titanic dataset.

**Prerequisites:** introductory Python; pandas, NumPy and Matplotlib.

Run this notebook from the TP1 folder, with `audit_of_political_engagement_14_2017.tab` and `titanic.csv` alongside it. On a hosted notebook service, upload both files to the working directory. The Brexit section contains worked examples and a few tasks; the Titanic section asks you to apply the same methods. Complete cells marked `TODO` in order.

## Part 1 — Guided example: Brexit ages

We introduce permutation testing through an analysis of the Brexit referendum, following [this Brexit age analysis](https://matthew-brett.github.io/les-pilot/brexit_ages.html). This also provides an opportunity to review pandas, a package for data manipulation that we will use throughout the course.

The Hansard Society survey interviewed 1771 people about the 2016 referendum. The local data file comes from the [Audit of Political Engagement 14](https://datacatalogue.ukdataservice.ac.uk/studies/study/8183).

In the sample, the average age of the Brexiteers is higher than that of the Remainers. Is this evidence of a population difference, or could it be explained by sampling variation?

We consider a **one-sided** hypothesis test:

$$H_0: \mu_B = \mu_R, \qquad H_1: \mu_B > \mu_R,$$

where $B$ denotes Brexiteers, $R$ denotes Remainers, and $\mu$ denotes the population mean age. Our statistic is

$$T_{\mathrm{obs}} = \operatorname{mean}(B)-\operatorname{mean}(R).$$

For the permutation test, under $H_0$ we assume that voting labels are exchangeable with respect to age: relabelling the pooled ages does not change their joint distribution. This motivates randomly permuting the labels while preserving the two group sizes. Equality of means alone does not guarantee exchangeability; this is the additional assumption used by this simple test.

Let's proceed in small steps, starting with pandas.

### Worked example — Data manipulation with pandas

In [ ]:
# Let's import the package 
import pandas as pd

In [ ]:
# Read the local tab-separated file into a DataFrame.
data_raw = pd.read_csv('audit_of_political_engagement_14_2017.tab', sep='\t')

Pandas represents tabular data as a **DataFrame**. Displaying it gives:

In [ ]:
data_raw

Typically, one would start by looking the first lines, using:

In [ ]:
data_raw.head()

Another useful pandas tool is `describe`, that is used to generate descriptive statistics of the data in a Pandas DataFrame

In [ ]:
data_raw.describe()

Each column is a **feature**, and each row represents one respondent. We need only `numage` (age) and `cut15` (referendum vote). Select these columns and give them descriptive names.

In [ ]:
# Feature selection
data = data_raw[['numage', 'cut15']]
# Rename columns
data.columns = ['age', 'vote']

The possible answer to the question - How did you vote on the question ‘Should the United Kingdom remain a member of the European Union or leave the European Union’?” - are encoded through $\textbf{labels}$, i.e. a number going from 1 to 6 encoding what was the answer:

Value = 1.0 $\qquad$ Label = Remain a member of the European Union

Value = 2.0 $\qquad$   Label = Leave the European Union

Value = 3.0 $\qquad$   Label = Did not vote

Value = 4.0 $\qquad$   Label = Too young

Value = 5.0 $\qquad$   Label = Can't remember

Value = 6.0 $\qquad$   Label = Refused

In [ ]:
# Let's see how it looks like now
data.head()

We can use `data.sort_values` to sort the dataset by age

In [ ]:
data.sort_values(by='age')

Keep valid ages and the two voting groups of interest. A zero in the age column represents an invalid age.

In [ ]:
data = data.dropna(subset=['age', 'vote'])
data = data[data['age'] > 0]
remainers = data[data['vote'] == 1]
brexiteers = data[data['vote'] == 2]
n_brexiteers = len(brexiteers)
n_remainers = len(remainers)
print(f"Brexiteers among these two groups: {n_brexiteers / (n_brexiteers + n_remainers):.1%}")

Let's have a look at the age distribution for the two. We import `matplotlib` for doing so, another well-known and important package that we shall use all the time.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
%matplotlib inline
fig, ax = plt.subplots(1,2, figsize=(20,7), sharex=True)
plt.suptitle("Age distribution", fontsize=16)
ax[0].set_title(f"Brexiteers", fontsize=14)
ax[0].hist(brexiteers['age'], color ='darkred')
ax[0].set_xlabel("Age", fontsize = 14)
ax[0].set_ylabel("# (people)", fontsize = 14)

ax[1].set_title(f"Remainers", fontsize=14)
ax[1].hist(remainers['age'], color ='darkblue')
ax[1].set_xlabel("Age", fontsize = 14)
ax[1].set_ylabel("# (people)", fontsize = 14)

Let's compute now the averages and compare them, using another well-known python package:

In [ ]:
import numpy as np

avg_brex = np.mean(brexiteers['age'])
avg_rem = np.mean(remainers['age'])
T_obs = avg_brex - avg_rem
print(f"Observed mean age difference (Brexiteers - Remainers): {T_obs:.3f} years")

### Worked example — One-sided permutation test

How unusual is the observed mean age difference under the exchangeability assumption?

The idea is simple:
- Randomly permute the labels, preserving the group sizes.
- Compute the mean age difference for each permutation.
- Count differences **at least as large as** the observed difference: `T_perm >= T_obs`.

Each permuted dataset has `n_brexiteers` ages in Group B and `n_remainers` ages in Group R. Shuffling the pooled ages and splitting at `n_brexiteers` is equivalent to permuting their labels.

In [ ]:
pooled_ages = np.concatenate((brexiteers['age'], remainers['age']))
print(f'Pooled sample size: {len(pooled_ages)}')

Use a fixed random seed to make the permutations reproducible.

In [ ]:
np.random.seed(123)

Generate 10,000 permutations and count those with a mean difference at least as large as the observed difference.

In [ ]:
n_permutations = 10_000
count = 0
for _ in range(n_permutations):
    shuffled = np.random.permutation(pooled_ages)
    shuffled_b = shuffled[:n_brexiteers]
    shuffled_r = shuffled[n_brexiteers:]
    T_perm = np.mean(shuffled_b) - np.mean(shuffled_r)
    if T_perm >= T_obs:
        count += 1
p_value = (count + 1) / (n_permutations + 1)

The Monte Carlo estimate is $p=(\mathrm{count}+1)/(n_{\mathrm{permutations}}+1)$. The correction includes the observed arrangement and avoids reporting zero. The p-value estimates the probability, under the permutation null, of a statistic at least as large as $T_{\mathrm{obs}}$; it is not the probability that $H_0$ is true.

In [ ]:
print(f"Permutations with T_perm >= T_obs: {count} / {n_permutations}")
print(f"One-sided Monte Carlo p-value: {p_value:.6f}")
print("Reject H0 at the 5% level." if p_value < 0.05 else "Do not reject H0 at the 5% level.")

A small p-value provides evidence for a higher population mean age among Brexiteers under the test assumptions. It does not establish a causal effect of age on voting.

### Bootstrap resampling

**Bootstrap** is a resampling strategy with replacement. It approximates the sampling distribution of a statistic by repeatedly drawing samples from the observed data, each of the original sample size.

It does not require a specified parametric distribution, but its accuracy still depends on the sample representing the population and on the sampling assumptions (here, independent observations). We will use it to estimate confidence intervals and the variance of an estimator.

### Exercise 1 — Compute sample medians

With one observed sample per group, how can we estimate uncertainty in the population median ages of Brexiteers and Remainers?

Let's recall the dataset and isolate the age feature.

To do so we take a single column of the dataset, which is an object called `Series`, and turn it into a `numpy.ndarray` using `pd.Series.to_numpy`

In [ ]:
brexit_ages = pd.Series.to_numpy(brexiteers['age'])
remain_ages = pd.Series.to_numpy(remainers['age'])

First of all we compute the median for the two groups.

Implement `median_value(array)` by sorting a copy of the input and handling odd and even sample sizes. Do not use `np.median` inside your function. Then print the sample median age of each group.

In [ ]:
# TODO: Implement median_value without np.median, handling odd and even lengths.

# YOUR CODE HERE

In [ ]:
# TODO: Print both sample median ages using your function.

# YOUR CODE HERE

Check your results against `np.median` after completing your function.

In [ ]:
# TODO: Check both results using np.median.

# YOUR CODE HERE

Now we use **bootstrap**: we repeatedly sample the two groups with replacement with the same sample size

First, we use Brexiteers as a worked example with 10,000 bootstrap repetitions.

In [ ]:
n = len(brexit_ages)
reps = 10_000

We then use 

```
np.random.choice(a, size=None, replace=True, p=None)
```

which generates a random sample of a certain size from a given 1-D array `a`. If `replace=True`, then a value of `a` can be selected multiple times.

In [ ]:
np.random.seed(123)
boot_brexit = np.random.choice(brexit_ages, (reps, n), replace=True)
boot_brexit_medians = np.median(boot_brexit, axis=1)

The results look like this

In [ ]:
boot_brexit[:5]

In [ ]:
boot_brexit_medians[:5]

The original sample median estimates the **population median**. The bootstrap medians form a **bootstrap distribution**, approximating how the sample median varies across samples. Its standard deviation estimates the standard error of the sample median.

In [ ]:
print(f"Mean of the bootstrap medians: {boot_brexit_medians.mean():.3f} years")

In [ ]:
print(f"Bootstrap standard error of the sample median: {boot_brexit_medians.std(ddof=1):.3f} years")

Use the 2.5th and 97.5th percentiles of the bootstrap distribution to construct an **approximate 95% bootstrap confidence interval for the population median age**. The 95% confidence level describes approximate coverage over repeated samples, not a probability assigned to the fixed population median.

In [ ]:
boot_brexit_median_CI = np.percentile(boot_brexit_medians, [2.5,97.5])
print(f"The C.I. for the median age of brexiteers computed with bootstrap is [{boot_brexit_median_CI[0]},{boot_brexit_median_CI[1]}] ")

### Exercise 2 — Bootstrap the Remainer median

Draw `reps` bootstrap samples of size `len(remain_ages)` with replacement. Store their medians in `boot_remain_medians`. Report their mean and standard deviation (the estimated standard error), and store the approximate 95% percentile interval in `boot_remain_median_CI`.

In [ ]:
# TODO: Resample Remainer ages and store boot_remain_medians.

# YOUR CODE HERE

In [ ]:
# TODO: Report bootstrap mean, standard error and boot_remain_median_CI.

# YOUR CODE HERE

The following plotting code compares the bootstrap distributions and their approximate 95% confidence intervals once you have completed Exercise 2.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(boot_brexit_medians, alpha=0.65, color='darkred', label='Brexiteers')
ax.hist(boot_remain_medians, alpha=0.65, color='darkblue', label='Remainers')
for index, bound in enumerate(boot_brexit_median_CI):
    ax.axvline(bound, color='darkred', linestyle='--', label='Brexiteers 95% interval' if index == 0 else None)
for index, bound in enumerate(boot_remain_median_CI):
    ax.axvline(bound, color='darkblue', linestyle='--', label='Remainers 95% interval' if index == 0 else None)
ax.set(title='Bootstrap distributions of sample median age (years)', xlabel='Median age (years)', ylabel='Bootstrap repetitions')
ax.legend();

Compare the interval locations and widths. Interval overlap alone is not a calibrated test of the difference between the population medians.

## Part 2 — Your turn: Titanic

The sinking of the Titanic is one of the most infamous shipwrecks in history.

On April 15, 1912, during her maiden voyage, the widely considered “unsinkable” RMS Titanic sank after colliding with an iceberg. Unfortunately, there weren’t enough lifeboats for everyone onboard, resulting in the death of the majority of passengers and crew.

While there was some element of luck involved in surviving, it seems some groups of people were more likely to survive than others.

**The data set:**

Load the local file `titanic.csv`, included alongside this notebook. Source: [Data Science Dojo Titanic dataset](https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv).

Apply the methods from Part 1. For each inferential result, state which population quantity you are estimating or testing.

### Age and ticket fares

### Exercise 3 — Load and inspect the data

- Import `titanic.csv` using `pd.read_csv` and display its first rows.
- How many columns are there? Here we count all columns, including identifiers and the survival outcome.

In [ ]:
# TODO: Load titanic.csv, display the first rows and report the number of columns.

# YOUR CODE HERE

### Exercise 4 — Clean age data and estimate survival

- Select `Age` and `Survived`.
- Remove rows with missing values in these columns using `dropna(how="any")`.
- Compute the percentage of survivors in this cleaned subset, an estimate of $P(Survived=1)$ among passengers with recorded ages.

In [ ]:
# TODO: Clean Age/Survived, create both groups and report the survival percentage.

# YOUR CODE HERE

### Exercise 5 — Compare age distributions

- Plot age histograms for survivors and non-survivors using common bins.
- Compute the difference in average ages (survivors minus non-survivors).

In [ ]:
# TODO: Plot labelled age histograms and compute survivors minus non-survivors mean age.

# YOUR CODE HERE

### Exercise 6 — Compare mean fares

- Starting from the full Titanic dataset, select `Fare` and `Survived` and remove missing values in these columns. Keep zero fares, which are recorded values.
- Form survivor and non-survivor groups and compute their mean fare difference (survivors minus non-survivors).

In [ ]:
# TODO: Clean Fare/Survived from the full data, form fare groups and compute their mean difference.

# YOUR CODE HERE

### Exercise 7 — Permutation test for mean fares

Use 50,000 permutations to test $H_0: \mu_S=\mu_N$ against the one-sided alternative $H_1: \mu_S>\mu_N$, where $S$ denotes survivors and $N$ non-survivors. Assume survival labels are exchangeable with respect to fare under the null.

Use $T_{\mathrm{obs}}=\operatorname{mean}(S)-\operatorname{mean}(N)$, preserve the original group sizes, count `T_perm >= T_obs`, and compute `(count + 1) / (n_permutations + 1)`. Report and interpret the p-value at the 5% level.

In [ ]:
# TODO: Run the one-sided fare permutation test and interpret its corrected p-value.

# YOUR CODE HERE

Interpret this as an association between fare and survival, without claiming a causal effect.

### Exercise 8 — Sample median fares

Compute and report the sample median fare for survivors and non-survivors.

In [ ]:
# TODO: Report the sample median fare in each survival group.

# YOUR CODE HERE

### Exercise 9 — Bootstrap confidence intervals for median fares

- For each group, draw 10,000 bootstrap samples with replacement, each of the original group size, using `np.random.choice`.
- Compute each resample's median.
- Use `np.percentile` to compute an approximate 95% bootstrap confidence interval for each population median fare.

In [ ]:
# TODO: Bootstrap both median fares and compute their approximate 95% percentile intervals.

# YOUR CODE HERE

### Exercise 10 — Visualize and interpret bootstrap distributions

- Plot the bootstrap median fare distributions, including their approximate 95% confidence intervals.
- Compare the locations and widths. What do they suggest about the population median fares?

In [ ]:
# TODO: Plot both bootstrap distributions with intervals and explain their locations and widths.

# YOUR CODE HERE

### Exercise 11 — Bootstrap variance of median estimators

Estimate the sampling variance of each sample median fare using the variance of its bootstrap medians. Distinguish this from the variance of individual passenger fares.

In [ ]:
# TODO: Estimate both median estimators' sampling variances from the bootstrap medians.

# YOUR CODE HERE

### Other features associated with survival

Now examine survival rates by **Sex** and **Pclass** (passenger class, with values 1, 2 and 3).

### Exercise 12 — Survival conditional on sex

Estimate $P(Survived=1\mid Sex=\mathrm{female})$ and $P(Survived=1\mid Sex=\mathrm{male})$ from the full dataset, dropping missing values only in the relevant columns.

In [ ]:
# TODO: Estimate survival probabilities conditional on each recorded sex.

# YOUR CODE HERE

### Exercise 13 — Survival conditional on sex and passenger class

- Estimate $P(Survived=1\mid Sex=\mathrm{male}, Pclass=c)$ for each $c\in\{1,2,3\}$.
- Identify the classes with the highest and lowest observed survival rates among men.
- Compare these two groups using 10,000 permutations of the binary survival outcomes, preserving group sizes. For this pair, use $H_0:p_A=p_B$, $H_1:p_A>p_B$, and $T_{\mathrm{obs}}=\operatorname{mean}(A)-\operatorname{mean}(B)$, where A is the group with the higher observed rate. Under the null, assume outcomes are exchangeable between the two classes. Count `T_perm >= T_obs` and use the corrected Monte Carlo p-value.
- Report and interpret the result.

Because the pair and direction are selected after looking at the data, this simple pairwise p-value is **exploratory**; it does not account for that selection. A confirmatory comparison should specify the pair and direction in advance.

In [ ]:
# TODO: Compute male survival by class, identify the extremes and run and interpret the exploratory permutation test.

# YOUR CODE HERE